# 📊 Exploration naïve des données

---
## 1. Chargement des données

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

print(f"✅ Pandas version: {pd.__version__}")
print(f"✅ Numpy version: {np.__version__}")

In [ ]:
# Chemins des fichiers
DATA_PATH = Path('../data/raw')
FAO_FILE = DATA_PATH / 'FAO.csv'
IMPACT_FILE = DATA_PATH / 'Food_Production.csv'

print(f"📂 Dossier données: {DATA_PATH.absolute()}")
print(f"✅ FAO.csv existe: {FAO_FILE.exists()}")
print(f"✅ Food_Production.csv existe: {IMPACT_FILE.exists()}")

---
## 2. Analyse FAO.csv - Production mondiale

### 2.1 Chargement et inspection initiale

In [ ]:
# Chargement FAO (peut prendre quelques secondes pour 4.4 MB)
print("⏳ Chargement FAO.csv...")

# Le fichier FAO.csv n'est pas en UTF-8, on essaie différents encodages
try:
    df_fao = pd.read_csv(FAO_FILE, encoding='latin-1')
    print("✅ Encodage détecté: latin-1")
except:
    try:
        df_fao = pd.read_csv(FAO_FILE, encoding='ISO-8859-1')
        print("✅ Encodage détecté: ISO-8859-1")
    except:
        df_fao = pd.read_csv(FAO_FILE, encoding='cp1252')
        print("✅ Encodage détecté: cp1252")

print(f"✅ Chargé: {len(df_fao):,} lignes × {len(df_fao.columns)} colonnes")

# Aperçu
df_fao.head()

### 2.2 Informations générales

In [ ]:
# Dimensions
print("=" * 60)
print("📊 DIMENSIONS & STRUCTURE")
print("=" * 60)
print(f"Nombre de lignes (observations): {len(df_fao):,}")
print(f"Nombre de colonnes (variables): {len(df_fao.columns)}")
print(f"\nListe des colonnes:")
for i, col in enumerate(df_fao.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Types de données
print("\n" + "=" * 60)
print("📋 TYPES DE DONNÉES")
print("=" * 60)
print(df_fao.dtypes.value_counts())
print("\nDétail:")
print(df_fao.dtypes)

In [ ]:
# Empreinte mémoire
print("\n" + "=" * 60)
print("💾 EMPREINTE MÉMOIRE")
print("=" * 60)

memory_usage = df_fao.memory_usage(deep=True)
total_mb = memory_usage.sum() / 1024**2

print(f"Mémoire totale: {total_mb:.2f} MB")
print(f"\nTop 5 colonnes les plus volumineuses:")
print(memory_usage.sort_values(ascending=False).head(5) / 1024**2)

In [ ]:
# Poids du fichier
file_size_mb = FAO_FILE.stat().st_size / 1024**2
print(f"\nTaille fichier sur disque: {file_size_mb:.2f} MB")
print(f"Ratio mémoire/disque: {total_mb/file_size_mb:.2f}x")

### 2.3 Qualité des données

In [ ]:
# Valeurs manquantes
print("=" * 60)
print("🔍 VALEURS MANQUANTES")
print("=" * 60)

missing = df_fao.isnull().sum()
missing_pct = (missing / len(df_fao)) * 100

missing_df = pd.DataFrame({
    'Colonnes': df_fao.columns,
    'Nb_nulls': missing.values,
    'Pct_nulls': missing_pct.values
}).sort_values('Nb_nulls', ascending=False)

print(missing_df[missing_df['Nb_nulls'] > 0])

print(f"\n✅ Colonnes sans nulls: {len(missing_df[missing_df['Nb_nulls'] == 0])}")
print(f"⚠️  Colonnes avec nulls: {len(missing_df[missing_df['Nb_nulls'] > 0])}")

In [ ]:
# Lignes dupliquées
print("\n" + "=" * 60)
print("🔄 LIGNES DUPLIQUÉES")
print("=" * 60)

duplicates = df_fao.duplicated().sum()
duplicates_pct = (duplicates / len(df_fao)) * 100

print(f"Nombre de lignes dupliquées: {duplicates:,} ({duplicates_pct:.2f}%)")

if duplicates > 0:
    print("\nExemple de lignes dupliquées:")
    print(df_fao[df_fao.duplicated(keep=False)].head())

In [ ]:
# Colonnes entièrement vides
print("\n" + "=" * 60)
print("🗑️  COLONNES VIDES")
print("=" * 60)

empty_cols = [col for col in df_fao.columns if df_fao[col].isnull().all()]

if empty_cols:
    print(f"Colonnes entièrement vides ({len(empty_cols)}):")
    for col in empty_cols:
        print(f"  - {col}")
else:
    print("✅ Aucune colonne entièrement vide")

### 2.4 Analyse descriptive

In [ ]:
# Colonnes catégorielles
print("=" * 60)
print("📝 VARIABLES CATÉGORIELLES")
print("=" * 60)

categorical_cols = df_fao.select_dtypes(include=['object']).columns

for col in categorical_cols:
    unique_count = df_fao[col].nunique()
    print(f"\n{col}:")
    print(f"  Valeurs uniques: {unique_count}")
    if unique_count <= 20:
        print(f"  Valeurs: {df_fao[col].unique()[:10].tolist()}")
    else:
        print(f"  Top 5 valeurs les plus fréquentes:")
        print(df_fao[col].value_counts().head())

In [ ]:
# Colonnes numériques (années Y1961-Y2013)
print("\n" + "=" * 60)
print("🔢 VARIABLES NUMÉRIQUES (ANNÉES)")
print("=" * 60)

# Identifier les colonnes d'années
year_cols = [col for col in df_fao.columns if col.startswith('Y')]
print(f"Nombre de colonnes temporelles: {len(year_cols)}")
print(f"Période couverte: {year_cols[0]} à {year_cols[-1]}")
print(f"\nStatistiques descriptives (première année {year_cols[0]}):")
print(df_fao[year_cols[0]].describe())

In [ ]:
# Distribution des valeurs sur toutes les années
print("\n📊 Distribution globale des productions (toutes années):")

# Flatten toutes les valeurs d'années
all_values = df_fao[year_cols].values.flatten()
all_values_clean = all_values[~np.isnan(all_values)]

print(f"\nStatistiques globales:")
print(f"  Min: {all_values_clean.min():,.0f}")
print(f"  Max: {all_values_clean.max():,.0f}")
print(f"  Moyenne: {all_values_clean.mean():,.0f}")
print(f"  Médiane: {np.median(all_values_clean):,.0f}")
print(f"  Écart-type: {all_values_clean.std():,.0f}")

In [ ]:
# Visualisation distribution (échantillon)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(all_values_clean, bins=50, edgecolor='black')
axes[0].set_xlabel('Production (1000 tonnes)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des productions (toutes années)')
axes[0].set_yscale('log')

# Boxplot
axes[1].boxplot(all_values_clean, vert=True)
axes[1].set_ylabel('Production (1000 tonnes)')
axes[1].set_title('Boxplot des productions')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

### 2.5 Outliers

In [ ]:
# Détection outliers (méthode IQR)
print("=" * 60)
print("⚠️  DÉTECTION D'OUTLIERS (méthode IQR)")
print("=" * 60)

Q1 = np.percentile(all_values_clean, 25)
Q3 = np.percentile(all_values_clean, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (all_values_clean < lower_bound) | (all_values_clean > upper_bound)
outliers_count = outliers.sum()
outliers_pct = (outliers_count / len(all_values_clean)) * 100

print(f"Q1 (25%): {Q1:,.0f}")
print(f"Q3 (75%): {Q3:,.0f}")
print(f"IQR: {IQR:,.0f}")
print(f"\nBorne inférieure: {lower_bound:,.0f}")
print(f"Borne supérieure: {upper_bound:,.0f}")
print(f"\nNombre d'outliers: {outliers_count:,} ({outliers_pct:.2f}%)")

In [ ]:
# Identifier les pays/produits avec les plus fortes productions (outliers potentiels)
print("\n📊 Top 10 des productions les plus élevées:")

# Créer un dataframe avec valeur max par ligne
df_fao['Max_Production'] = df_fao[year_cols].max(axis=1)
top_producers = df_fao.nlargest(10, 'Max_Production')[['Area', 'Item', 'Element', 'Max_Production']]
print(top_producers)

---
## 3. Analyse Food_Production.csv - Impact environnemental

### 3.1 Chargement et inspection

In [ ]:
# Chargement
print("⏳ Chargement Food_Production.csv...")
df_impact = pd.read_csv(IMPACT_FILE)
print(f"✅ Chargé: {len(df_impact):,} lignes × {len(df_impact.columns)} colonnes")

# Aperçu
df_impact.head(10)

In [ ]:
# Informations générales
print("=" * 60)
print("📊 STRUCTURE Food_Production.csv")
print("=" * 60)
print(f"Dimensions: {df_impact.shape[0]} lignes × {df_impact.shape[1]} colonnes")
print(f"\nListe des colonnes:")
for i, col in enumerate(df_impact.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Types et mémoire
print("\n" + "=" * 60)
print("📋 TYPES DE DONNÉES")
print("=" * 60)
print(df_impact.dtypes)

impact_memory = df_impact.memory_usage(deep=True).sum() / 1024
print(f"\n💾 Mémoire totale: {impact_memory:.2f} KB")

### 3.2 Qualité des données

In [ ]:
# Valeurs manquantes
print("=" * 60)
print("🔍 QUALITÉ DES DONNÉES")
print("=" * 60)

missing_impact = df_impact.isnull().sum()
missing_impact_pct = (missing_impact / len(df_impact)) * 100

print("\nValeurs manquantes par colonne:")
for col in df_impact.columns:
    nulls = missing_impact[col]
    pct = missing_impact_pct[col]
    if nulls > 0:
        print(f"  {col:50s}: {nulls:3d} ({pct:5.1f}%)")

print(f"\nLignes dupliquées: {df_impact.duplicated().sum()}")

### 3.3 Analyse des métriques d'impact

In [ ]:
# Liste des produits
print("=" * 60)
print("🥗 PRODUITS ALIMENTAIRES")
print("=" * 60)
print(f"Nombre de produits: {df_impact['Food product'].nunique()}")
print(f"\nListe complète:")
for i, product in enumerate(df_impact['Food product'].values, 1):
    print(f"  {i:2d}. {product}")

In [ ]:
# Colonnes d'impact disponibles
print("\n" + "=" * 60)
print("🌍 MÉTRIQUES D'IMPACT ENVIRONNEMENTAL")
print("=" * 60)

impact_cols = [
    'Total_emissions',  # GES total
    'Greenhouse gas emissions per kilogram (kgCO₂eq per kilogram)',
    'Freshwater withdrawals per kilogram (liters per kilogram)',
    'Land use per kilogram (m² per kilogram)',
    'Eutrophying emissions per kilogram (gPO₄eq per kilogram)'
]

# Vérifier disponibilité
for col in impact_cols:
    if col in df_impact.columns:
        non_null = df_impact[col].notna().sum()
        print(f"✅ {col:60s}: {non_null} valeurs")
    else:
        print(f"❌ {col:60s}: NON DISPONIBLE")

In [ ]:
# Statistiques descriptives des impacts
print("\n📊 Statistiques descriptives (Total_emissions - kgCO₂eq):")
print(df_impact['Total_emissions'].describe())

# Top/Bottom produits par émission
print("\n🔝 Top 5 produits avec le plus d'émissions CO₂:")
print(df_impact.nlargest(5, 'Total_emissions')[['Food product', 'Total_emissions']])

print("\n✅ Top 5 produits avec le moins d'émissions CO₂:")
print(df_impact.nsmallest(5, 'Total_emissions')[['Food product', 'Total_emissions']])

In [ ]:
# Visualisation impact par produit
fig, ax = plt.subplots(figsize=(12, 8))

# Trier par émissions
df_sorted = df_impact.sort_values('Total_emissions', ascending=True)

ax.barh(df_sorted['Food product'], df_sorted['Total_emissions'])
ax.set_xlabel('Total Emissions (kgCO₂eq per kg)', fontsize=12)
ax.set_ylabel('Food Product', fontsize=12)
ax.set_title('Impact Carbone par Produit Alimentaire', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Mapping des produits FAO ↔ Impact

**Objectif**: Identifier la correspondance entre les noms de produits dans FAO.csv et Food_Production.csv

In [ ]:
# Liste des produits FAO
print("=" * 60)
print("🔗 MAPPING PRODUITS FAO ↔ IMPACT")
print("=" * 60)

fao_products = df_fao['Item'].unique()
impact_products = df_impact['Food product'].unique()

print(f"Produits FAO: {len(fao_products)}")
print(f"Produits Impact: {len(impact_products)}")

print("\n📋 Échantillon produits FAO (20 premiers):")
for i, prod in enumerate(sorted(fao_products)[:20], 1):
    print(f"  {i:2d}. {prod}")

In [ ]:
# Recherche de correspondances (matching simple par mots-clés)
print("\n🔍 Recherche de correspondances automatiques...\n")

# Mapping manuel à affiner
mapping_dict = {
    'Wheat': ['Wheat', 'Wheat & Rye'],
    'Rice': ['Rice', 'Rice (Milled Equivalent)'],
    'Barley': ['Barley', 'Barley and products'],
    'Maize': ['Maize', 'Maize (Meal)'],
    'Oats': ['Oatmeal'],
    'Potatoes': ['Potatoes'],
    'Sugar': ['Cane Sugar', 'Beet Sugar'],
    'Soybeans': ['Soybean Oil', 'Soymilk', 'Tofu'],
    'Groundnuts': ['Groundnuts'],
    'Palm': ['Palm Oil'],
    'Sunflower': ['Sunflower Oil'],
    'Rapeseed': ['Rapeseed Oil']
}

print("🎯 Correspondances identifiées (à affiner):")
for fao_key, impact_list in mapping_dict.items():
    print(f"\n  FAO: {fao_key}")
    for imp in impact_list:
        print(f"    → Impact: {imp}")

In [ ]:
# Créer une table de correspondance à affiner manuellement
print("\n⚠️  ACTIONS NÉCESSAIRES:")
print("""
1. Créer une table de correspondance manuelle FAO ↔ Impact
2. Certains produits FAO n'ont pas d'équivalent dans Impact (ex: produits transformés)
3. Certains produits Impact sont des catégories (ex: "Wheat & Rye")
4. Solution: Créer un fichier CSV de mapping dans data/raw/product_mapping.csv

Colonnes suggérées:
  - fao_item: Nom dans FAO.csv
  - impact_product: Nom dans Food_Production.csv
  - category: Catégorie (céréales, viande, légumes, etc.)
  - match_quality: exact, approximate, missing
""")

---
## 5. Architecture Base de Données

### 5.1 Système OLAP vs OLTP

In [ ]:
print("=" * 60)
print("🏗️  ARCHITECTURE BASE DE DONNÉES")
print("=" * 60)

print("""
📊 SYSTÈME RECOMMANDÉ: OLAP (Online Analytical Processing)

Justification:
- Données historiques (1961-2023) = lecture seule
- Analyses multidimensionnelles (pays × produit × année)
- Agrégations complexes (SUM, AVG par dimensions)
- Pas de transactions fréquentes (OLTP)

Modèle: Schéma en étoile (Star Schema)
""")

### 5.2 Tables de Fait et Dimensions

In [ ]:
print("\n📐 SCHÉMA PROPOSÉ (Star Schema)\n")

schema = """
┌─────────────────────────────────────────────────────┐
│           TABLES DE FAIT (Fact Tables)             │
└─────────────────────────────────────────────────────┘

1. Fait_Production
   ├── production_id (PK)
   ├── pays_id (FK → Dim_Pays)
   ├── produit_id (FK → Dim_Produits)
   ├── annee_id (FK → Dim_Temps)
   ├── element (Food/Feed)
   └── quantite_tonnes (NUMERIC)

2. Fait_Impact
   ├── impact_id (PK)
   ├── produit_id (FK → Dim_Produits)
   ├── co2_per_kg (NUMERIC)
   ├── eau_litres_per_kg (NUMERIC)
   ├── land_m2_per_kg (NUMERIC)
   ├── eutrophication_per_kg (NUMERIC)
   └── ... (autres métriques)

┌─────────────────────────────────────────────────────┐
│        TABLES DE DIMENSION (Dimension Tables)      │
└─────────────────────────────────────────────────────┘

3. Dim_Pays
   ├── pays_id (PK)
   ├── nom_pays (VARCHAR)
   ├── code_iso (VARCHAR)
   ├── region (VARCHAR)
   ├── latitude (NUMERIC)
   └── longitude (NUMERIC)

4. Dim_Produits
   ├── produit_id (PK)
   ├── nom_produit (VARCHAR)
   ├── categorie (VARCHAR) [céréales, viande, légumes...]
   └── sous_categorie (VARCHAR)

5. Dim_Temps
   ├── annee_id (PK)
   ├── annee (INTEGER)
   ├── decennie (INTEGER)
   └── periode (VARCHAR) [1960s, 1970s...]

6. Dim_Socio_Economique (enrichissement)
   ├── socio_id (PK)
   ├── pays_id (FK → Dim_Pays)
   ├── annee_id (FK → Dim_Temps)
   ├── pib_per_capita (NUMERIC)
   ├── taux_urbanisation (NUMERIC)
   └── population (NUMERIC)
"""

print(schema)

### 5.3 Granularité des données

In [ ]:
print("\n📏 GRANULARITÉ DES DONNÉES\n")

# Vérifier si df_fao existe
if 'df_fao' in locals():
    # Identifier les colonnes d'années
    year_cols = [col for col in df_fao.columns if col.startswith('Y')]
    
    print("Fait_Production:")
    print("  - Niveau: Pays × Produit × Année × Element")
    print(f"  - Pays uniques: {df_fao['Area'].nunique()}")
    print(f"  - Produits uniques: {df_fao['Item'].nunique()}")
    print(f"  - Années: {len(year_cols)} (1961-2013)")
    print(f"  - Elements: {df_fao['Element'].nunique()}")
else:
    print("⚠️  Veuillez d'abord exécuter les cellules de chargement FAO")

print("\nFait_Impact:")
if 'df_impact' in locals():
    print("  - Niveau: Produit (statique, pas de dimension temporelle)")
    print(f"  - Produits uniques: {df_impact['Food product'].nunique()}")
else:
    print("⚠️  Veuillez d'abord exécuter les cellules de chargement Food_Production")

---
## 6. Variable Cible et ML

### 6.1 Définition de la variable cible

In [ ]:
print("=" * 60)
print("🎯 VARIABLE CIBLE")
print("=" * 60)

print("""
Variable cible proposée:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📌 Empreinte Carbone Totale par Pays/Année

Formule:
  Impact_CO2_total = Σ (production_tonnes × 1000 × co2_per_kg)
  
Où:
  - production_tonnes: Production FAO en milliers de tonnes
  - co2_per_kg: Émissions CO₂ par kg (Food_Production.csv)
  - ×1000: Conversion milliers de tonnes → kg

Exemple:
  France 2010, Blé:
  - Production: 38,000 (milliers de tonnes) = 38,000,000 kg
  - Impact blé: 1.4 kgCO₂eq/kg
  → Impact total blé: 38,000,000 × 1.4 = 53,200,000 kgCO₂eq

Unité finale: kgCO₂eq ou tonnes CO₂eq

Type de variable: Continue (régression)
""")

### 6.2 Type de problématique ML

In [ ]:
print("\n" + "=" * 60)
print("🤖 PROBLÉMATIQUES MACHINE LEARNING")
print("=" * 60)

print("""
Problématique 1: RÉGRESSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Objectif: Prédire l'empreinte carbone d'un pays selon son mix alimentaire

Variable cible:
  - Impact_CO2_total (continu)
  
Features (variables explicatives):
  - Production par catégorie (céréales, viande, légumes...)
  - PIB par habitant
  - Taux d'urbanisation
  - Population
  - Année (tendance temporelle)
  
Modèles pressentis:
  1. Régression Linéaire (baseline)
  2. Random Forest Regressor
  3. XGBoost Regressor ⭐ (recommandé)
  4. Ridge/Lasso (si multicolinéarité)

Métriques d'évaluation:
  - R² (coefficient de détermination)
  - RMSE (Root Mean Squared Error)
  - MAE (Mean Absolute Error)
  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Problématique 2: CLUSTERING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Objectif: Profiler les pays selon leur régime alimentaire et impact

Features:
  - % production céréales / viande / légumes
  - Impact CO₂ moyen
  - Impact hydrique moyen
  - PIB, urbanisation
  
Modèles pressentis:
  1. KMeans (k=3-7 clusters)
  2. DBSCAN (détection clusters de densité)
  3. Hierarchical Clustering

Métriques d'évaluation:
  - Silhouette Score
  - Davies-Bouldin Index
  - Inertia (KMeans)
  
Visualisation:
  - PCA 2D/3D pour visualiser les clusters
  - Profils moyens par cluster
""")

### 6.3 Réalisabilité de la problématique

In [ ]:
print("\n" + "=" * 60)
print("✅ RÉALISABILITÉ")
print("=" * 60)

print("""
La problématique EST réalisable car:

✅ Données disponibles:
   - Production mondiale détaillée (FAO)
   - Impact environnemental par produit (Food_Production)
   - Possibilité d'enrichir (World Bank, OWID)

✅ Variable cible claire:
   - Calculable: production × impact = empreinte
   - Continue (régression)
   - Interprétable

✅ Features pertinentes:
   - Production par catégorie
   - Indicateurs socio-économiques
   - Dimension temporelle

✅ Volume de données suffisant:
   - FAO: ~100K lignes (pays × produits × années)
   - Après transformation: ~250K observations
   - Suffisant pour ML

⚠️  Défis identifiés:
   1. Mapping FAO ↔ Impact à finaliser
   2. Données FAO s'arrêtent en 2013 (enrichir 2014-2023)
   3. Valeurs manquantes à gérer (imputation)
   4. Outliers (grandes productions) à traiter
""")

---
## 7. Conclusions et prochaines étapes

### 7.1 Synthèse de l'exploration

In [ ]:
print("=" * 60)
print("📝 SYNTHÈSE DE L'EXPLORATION NAÏVE")
print("=" * 60)

# Calculer les statistiques si les variables n'existent pas
if 'df_fao' in locals():
    # Recalculer si nécessaire
    if 'year_cols' not in locals():
        year_cols = [col for col in df_fao.columns if col.startswith('Y')]
    if 'total_mb' not in locals():
        total_mb = df_fao.memory_usage(deep=True).sum() / 1024**2
    if 'duplicates' not in locals():
        duplicates = df_fao.duplicated().sum()
    if 'outliers_pct' not in locals():
        all_values = df_fao[year_cols].values.flatten()
        all_values_clean = all_values[~np.isnan(all_values)]
        Q1 = np.percentile(all_values_clean, 25)
        Q3 = np.percentile(all_values_clean, 75)
        IQR = Q3 - Q1
        outliers = (all_values_clean < Q1 - 1.5 * IQR) | (all_values_clean > Q3 + 1.5 * IQR)
        outliers_pct = (outliers.sum() / len(all_values_clean)) * 100
    
    print(f"""
1. FAO.csv - Production mondiale
   ├── Lignes: {len(df_fao):,}
   ├── Colonnes: {len(df_fao.columns)}
   ├── Pays: {df_fao['Area'].nunique()}
   ├── Produits: {df_fao['Item'].nunique()}
   ├── Période: 1961-2013 ({len(year_cols)} années)
   ├── Mémoire: {total_mb:.2f} MB
   ├── Duplicatas: {duplicates:,}
   └── Outliers: {outliers_pct:.1f}%
""")
else:
    print("\n⚠️  FAO.csv non chargé - Exécutez d'abord la section 2\n")

if 'df_impact' in locals():
    if 'impact_memory' not in locals():
        impact_memory = df_impact.memory_usage(deep=True).sum() / 1024
    
    print(f"""
2. Food_Production.csv - Impact environnemental
   ├── Lignes: {len(df_impact)}
   ├── Colonnes: {len(df_impact.columns)}
   ├── Produits: {df_impact['Food product'].nunique()}
   ├── Métriques: CO₂, eau, terres, eutrophisation
   └── Qualité: Valeurs manquantes à gérer
""")
else:
    print("⚠️  Food_Production.csv non chargé - Exécutez d'abord la section 3\n")

print("""
3. Architecture recommandée
   ├── Système: OLAP (analyse multidimensionnelle)
   ├── Modèle: Schéma en étoile
   ├── Tables de fait: Production, Impact
   └── Dimensions: Pays, Produits, Temps, Socio-économique

4. Machine Learning
   ├── Régression: Prédire empreinte carbone
   │   └── Modèles: RandomForest, XGBoost, Ridge
   └── Clustering: Profiler pays
       └── Modèles: KMeans, DBSCAN
""")

### 7.2 Actions à entreprendre

In [ ]:
print("\n" + "=" * 60)
print("🚀 PROCHAINES ÉTAPES")
print("=" * 60)

print("""
Phase 1: Enrichissement données (2-3h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Créer notebook enrichissement_donnees.ipynb
□ Récupérer production FAO 2014-2022 (API FAOSTAT)
□ Récupérer PIB, urbanisation (World Bank API)
□ Récupérer consommation viande (OWID GitHub)
□ Créer table de mapping produits FAO ↔ Impact

Phase 2: Architecture DB (1h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Modéliser schéma sur DBDiagram.io
□ Créer Data Lake (DuckDB/PostgreSQL)
□ Définir tables DWH (fait + dimensions)

Phase 3: Pipeline ETL (4h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Notebook 02_etl_pipeline.ipynb
□ Unpivot données FAO (années en lignes)
□ Jointure Production × Impact
□ Calcul empreinte carbone totale
□ Gestion nulls, outliers
□ Ingestion vers DWH
□ Export script etl_pipeline.py

Phase 4: EDA approfondie (8h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Notebook 03_eda_avancee.ipynb
□ Analyses uni/bi/multi-variées
□ Tests statistiques (ANOVA, corrélations)
□ Visualisations avancées
□ Conclusion: features pour ML

Phase 5: Machine Learning (6h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Notebook 04_machine_learning.ipynb
□ Preprocessing (split, encoding, scaling)
□ Régression (RandomForest, XGBoost)
□ Clustering (KMeans, DBSCAN)
□ Fine-tuning, MLFlow tracking
□ Sauvegarde modèles (.pkl)

Phase 6: Application Streamlit (6h)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
□ Créer app/streamlit_app.py
□ Simulateur menu
□ Comparateur pays
□ Prédictions
□ Déploiement
""")

### 7.3 Sauvegarde des résultats

In [ ]:
# Sauvegarder les statistiques clés
import json
from pathlib import Path

summary = {}

# Statistiques FAO
if 'df_fao' in locals():
    # Recalculer les variables si nécessaire
    if 'year_cols' not in locals():
        year_cols = [col for col in df_fao.columns if col.startswith('Y')]
    if 'total_mb' not in locals():
        total_mb = df_fao.memory_usage(deep=True).sum() / 1024**2
    if 'duplicates' not in locals():
        duplicates = df_fao.duplicated().sum()
    if 'outliers_pct' not in locals():
        all_values = df_fao[year_cols].values.flatten()
        all_values_clean = all_values[~np.isnan(all_values)]
        Q1 = np.percentile(all_values_clean, 25)
        Q3 = np.percentile(all_values_clean, 75)
        IQR = Q3 - Q1
        outliers = (all_values_clean < Q1 - 1.5 * IQR) | (all_values_clean > Q3 + 1.5 * IQR)
        outliers_pct = (outliers.sum() / len(all_values_clean)) * 100
    
    summary['fao'] = {
        'rows': len(df_fao),
        'columns': len(df_fao.columns),
        'countries': df_fao['Area'].nunique(),
        'products': df_fao['Item'].nunique(),
        'years': len(year_cols),
        'memory_mb': float(total_mb),
        'duplicates': int(duplicates),
        'outliers_pct': float(outliers_pct)
    }
    print("✅ Statistiques FAO calculées")
else:
    print("⚠️  FAO.csv non chargé - Statistiques FAO non disponibles")

# Statistiques Impact
if 'df_impact' in locals():
    if 'impact_memory' not in locals():
        impact_memory = df_impact.memory_usage(deep=True).sum() / 1024
    
    summary['impact'] = {
        'rows': len(df_impact),
        'columns': len(df_impact.columns),
        'products': df_impact['Food product'].nunique(),
        'memory_kb': float(impact_memory)
    }
    print("✅ Statistiques Impact calculées")
else:
    print("⚠️  Food_Production.csv non chargé - Statistiques Impact non disponibles")

# Sauvegarder en JSON
if summary:
    output_path = Path('../docs/exploration_summary.json')
    output_path.parent.mkdir(exist_ok=True)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Résumé sauvegardé: {output_path}")
    print(f"📊 Contenu: {list(summary.keys())}")
else:
    print("\n⚠️  Aucune donnée à sauvegarder - Exécutez d'abord les sections 2 et 3")